# Instella Reasoning — Reliability Suite (Colab GPU)

Runs `experiments/run_reliability_suite.sh` — the scaled-down reliability run: **gsm8k, 120 base
items × (answer-preserving + numeric) variants, 4 models**, generated over the *variant clusters*
so **Reliability = Accuracy × Consistency** is genuinely measured. Auto-saves to the HF dataset
`GOVINDFROM/Instella-Reasoning` as it runs (resume-safe across VM recycles).

**Prereqs:** Runtime → GPU. Colab Secrets (🔑): `github` (repo PAT) + `hf` (HF token, **WRITE** access).

**Non-T4 / Blackwell GPUs:** bitsandbytes 4-bit emits degenerate output on Blackwell, so Cell 1
patches the suite to run **bf16** (you have the VRAM). The degenerate *gate* is relaxed to a warning
so the weak 1B base model doesn't abort the run. Neither patch changes *what* is measured.

## Cell 1 — Setup (clone `main`, install, patch to bf16)

In [ ]:
%cd /content
!rm -rf Instella-Reasoning

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")   # GitHub PAT (repo read)
os.environ["HF_TOKEN"] = userdata.get("hf")            # HF token, WRITE access

!git clone https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning
%cd Instella-Reasoning
!git checkout main && git pull

assert os.path.exists("experiments/run_reliability_suite.sh"), "clone didn't get main"
print("OK: reliability suite present")

!pip install -e ".[hf,retrieval,viz,stats]"
!pip install "bitsandbytes>=0.46.1"

# Patch 1 - run bf16 instead of 4-bit (bitsandbytes 4-bit is broken on Blackwell GPUs)
!sed -i 's/prec="--load-in-4bit"/prec=""/' experiments/run_reliability_suite.sh
# Patch 2 - relax the degenerate hard-abort to a warning (weak 1B base loops naturally)
!sed -i 's/--fail-degenerate 0.25/--fail-degenerate 1.0/' experiments/run_reliability_suite.sh
!grep -n 'prec=\|fail-degenerate' experiments/run_reliability_suite.sh   # expect: prec=""  and  --fail-degenerate 1.0

## Cell 2 — Run the full 120-item suite
Uses a **fresh `OUT` dir** so no stale prep is restored from HF (forces a true 120). ~1–2 h on a fast
GPU. Resumable: if the VM recycles, just re-run this cell — it pulls finished work back from HF and
continues. Confirm it's the full run by seeing **`Wrote 556 generations`** (not 66).

In [ ]:
!rm -f data/processed/gsm8k.jsonl
!OUT=experiments/runs/reliability-B120-K5-clean bash experiments/run_reliability_suite.sh

## Cell 3 — Status / accuracy (run anytime)

In [ ]:
import glob, json, os
D = "experiments/runs/reliability-B120-K5-clean"
for f in sorted(glob.glob(f"{D}/scores/*__gsm8k.jsonl")):
    rows = [json.loads(l) for l in open(f) if l.strip()]
    acc = sum(r.get("correct", 0) for r in rows) / len(rows) if rows else 0
    print(f"{os.path.basename(f):34} rows={len(rows):4}  base_acc={acc:.3f}")